In [1]:
!pip install pandas sqlalchemy pymysql requests

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import requests
import numpy as np

from sqlalchemy import create_engine, text
from datetime import datetime

print("All Libraries imported")

All Libraries imported


## MySQL Database Setup

### 1. Create Database

```sql
CREATE DATABASE etl_project;

USE etl_project;
```

### 2. Create Customers Table

```sql
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    name VARCHAR(100),
    email VARCHAR(150),
    city VARCHAR(100),
    updated_at DATETIME,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
```

### 3. Verify Table

```sql
SHOW TABLES;

DESCRIBE customers;

SELECT * FROM customers;
```

**Database:** `etl_project`

**Table:** `customers`

**Primary Key:** `customer_id`

**Purpose:** Store customer records and prevent duplicates using incremental data loading and UPSERT operations.

In [49]:
username="Developer"
password = "Atharva9860866"
host="localhost"
database = "etl_project"

connection_string = (
    f"mysql+pymysql://{username}:{password}@{host}/{database}"
)

engine = create_engine(connection_string)

print("MySQL connection created successfully!")

MySQL connection created successfully!


In [8]:
data = {
    "customer_id": [
        101, 102, 103, 104, 105, 106, 107, 108, 109, 110,
        111, 112, 113, 114, 115, 116, 117, 118, 119, 120,
        121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
        131, 132, 133, 134, 135, 136, 137, 138, 139, 140,
        141, 142, 143, 144, 145, 146, 147, 148, 149, 150
    ],

    "name": [
        "Rahul", "Atharva", "Mohit", "Priya", "Amit",
        "Sneha", "Rohit", "Neha", "Karan", "Pooja",
        "Vikas", "Anjali", "Saurabh", "Riya", "Akash",
        "Kavita", "Nikhil", "Shreya", "Varun", "Meera",
        "Aditya", "Isha", "Manish", "Simran", "Arjun",
        "Divya", "Harsh", "Tanvi", "Gaurav", "Aarti",
        "Siddharth", "Komal", "Yash", "Payal", "Abhishek",
        "Nikita", "Rajat", "Swati", "Deepak", "Mansi",
        "Pranav", "Rashmi", "Aniket", "Pallavi", "Kunal",
        "Sakshi", "Tushar", "Naina", "Varsha", "Omkar"
    ],

    "email": [
        "rahul@gmail.com", "atharva@gmail.com", "mohit@gmail.com",
        "priya@gmail.com", "amit@gmail.com", "sneha@gmail.com",
        "rohit@gmail.com", "neha@gmail.com", "karan@gmail.com",
        "pooja@gmail.com", "vikas@gmail.com", "anjali@gmail.com",
        "saurabh@gmail.com", "riya@gmail.com", "akash@gmail.com",
        "kavita@gmail.com", "nikhil@gmail.com", "shreya@gmail.com",
        "varun@gmail.com", "meera@gmail.com", "aditya@gmail.com",
        "isha@gmail.com", "manish@gmail.com", "simran@gmail.com",
        "arjun@gmail.com", "divya@gmail.com", "harsh@gmail.com",
        "tanvi@gmail.com", "gaurav@gmail.com", "aarti@gmail.com",
        "siddharth@gmail.com", "komal@gmail.com", "yash@gmail.com",
        "payal@gmail.com", "abhishek@gmail.com", "nikita@gmail.com",
        "rajat@gmail.com", "swati@gmail.com", "deepak@gmail.com",
        "mansi@gmail.com", "pranav@gmail.com", "rashmi@gmail.com",
        "aniket@gmail.com", "pallavi@gmail.com", "kunal@gmail.com",
        "sakshi@gmail.com", "tushar@gmail.com", "naina@gmail.com",
        "varsha@gmail.com", "omkar@gmail.com"
    ],

    "city": [
        "Pune", "Mumbai", "Delhi", "Bangalore", "Hyderabad",
        "Chennai", "Kolkata", "Jaipur", "Ahmedabad", "Surat",
        "Nagpur", "Nashik", "Indore", "Bhopal", "Lucknow",
        "Kanpur", "Patna", "Ranchi", "Noida", "Gurgaon",
        "Pune", "Mumbai", "Delhi", "Bangalore", "Hyderabad",
        "Chennai", "Kolkata", "Jaipur", "Ahmedabad", "Surat",
        "Nagpur", "Nashik", "Indore", "Bhopal", "Lucknow",
        "Kanpur", "Patna", "Ranchi", "Noida", "Gurgaon",
        "Pune", "Mumbai", "Delhi", "Bangalore", "Hyderabad",
        "Chennai", "Kolkata", "Jaipur", "Ahmedabad", "Surat"
    ],

    "updated_at": [
        "2026-11-01 10:00:00",
        "2026-11-01 11:00:00",
        "2026-11-02 12:00:00",
        "2026-11-02 13:00:00",
        "2026-11-02 14:00:00",
        "2026-11-02 15:00:00",
        "2026-11-03 09:00:00",
        "2026-11-03 10:00:00",
        "2026-11-03 11:00:00",
        "2026-11-03 12:00:00",
        "2026-11-03 13:00:00",
        "2026-11-03 14:00:00",
        "2026-11-03 15:00:00",
        "2026-11-03 16:00:00",
        "2026-11-03 17:00:00",
        "2026-11-03 18:00:00",
        "2026-11-03 19:00:00",
        "2026-11-03 20:00:00",
        "2026-11-03 21:00:00",
        "2026-11-03 22:00:00",
        "2026-11-04 09:00:00",
        "2026-11-04 10:00:00",
        "2026-11-04 11:00:00",
        "2026-11-04 12:00:00",
        "2026-11-04 13:00:00",
        "2026-11-04 14:00:00",
        "2026-11-04 15:00:00",
        "2026-11-04 16:00:00",
        "2026-11-04 17:00:00",
        "2026-11-04 18:00:00",
        "2026-11-05 09:00:00",
        "2026-11-05 10:00:00",
        "2026-11-05 11:00:00",
        "2026-11-05 12:00:00",
        "2026-11-05 13:00:00",
        "2026-11-05 14:00:00",
        "2026-11-05 15:00:00",
        "2026-11-05 16:00:00",
        "2026-11-05 17:00:00",
        "2026-11-05 18:00:00",
        "2026-11-06 09:00:00",
        "2026-11-06 10:00:00",
        "2026-11-06 11:00:00",
        "2026-11-06 12:00:00",
        "2026-11-06 13:00:00",
        "2026-11-06 14:00:00",
        "2026-11-06 15:00:00",
        "2026-11-06 16:00:00",
        "2026-11-06 17:00:00",
        "2026-11-06 18:00:00"
    ]
}

df_csv = pd.DataFrame(data)

df_csv.to_csv("customers.csv",index=False)

print("CSV File Created Successfully")

CSV File Created Successfully


In [9]:
df_csv = pd.read_csv("customers.csv")

print(df_csv)

    customer_id       name                email       city  \
0           101      Rahul      rahul@gmail.com       Pune   
1           102    Atharva    atharva@gmail.com     Mumbai   
2           103      Mohit      mohit@gmail.com      Delhi   
3           104      Priya      priya@gmail.com  Bangalore   
4           105       Amit       amit@gmail.com  Hyderabad   
5           106      Sneha      sneha@gmail.com    Chennai   
6           107      Rohit      rohit@gmail.com    Kolkata   
7           108       Neha       neha@gmail.com     Jaipur   
8           109      Karan      karan@gmail.com  Ahmedabad   
9           110      Pooja      pooja@gmail.com      Surat   
10          111      Vikas      vikas@gmail.com     Nagpur   
11          112     Anjali     anjali@gmail.com     Nashik   
12          113    Saurabh    saurabh@gmail.com     Indore   
13          114       Riya       riya@gmail.com     Bhopal   
14          115      Akash      akash@gmail.com    Lucknow   
15      

In [11]:
df_csv.head()

,customer_id,name,email,city,updated_at
0,101,Rahul,rahul@gmail.com,Pune,2026-11-01 10:00:00
1,102,Atharva,atharva@gmail.com,Mumbai,2026-11-01 11:00:00
2,103,Mohit,mohit@gmail.com,Delhi,2026-11-02 12:00:00
3,104,Priya,priya@gmail.com,Bangalore,2026-11-02 13:00:00
4,105,Amit,amit@gmail.com,Hyderabad,2026-11-02 14:00:00


In [12]:
df_csv.tail()

,customer_id,name,email,city,updated_at
45,146,Sakshi,sakshi@gmail.com,Chennai,2026-11-06 14:00:00
46,147,Tushar,tushar@gmail.com,Kolkata,2026-11-06 15:00:00
47,148,Naina,naina@gmail.com,Jaipur,2026-11-06 16:00:00
48,149,Varsha,varsha@gmail.com,Ahmedabad,2026-11-06 17:00:00
49,150,Omkar,omkar@gmail.com,Surat,2026-11-06 18:00:00


# Extract a data From API

In [16]:
api_url = "https://dummyjson.com/users?limit=10"

response = requests.get(api_url)

response.raise_for_status()

api_data = response.json()

df_api = pd.DataFrame(api_data['users'])

print(df_api.head(5))

   id firstName  lastName maidenName  age  gender  \
0   1     Emily   Johnson      Smith   29  female   
1   2   Michael  Williams              36    male   
2   3    Sophia     Brown              43  female   
3   4     James     Davis              46    male   
4   5      Emma    Miller    Johnson   31  female   

                              email             phone  username      password  \
0     emily.johnson@x.dummyjson.com  +81 965-431-3024    emilys    emilyspass   
1  michael.williams@x.dummyjson.com  +49 258-627-6644  michaelw  michaelwpass   
2      sophia.brown@x.dummyjson.com  +81 210-652-2785   sophiab   sophiabpass   
3       james.davis@x.dummyjson.com  +49 614-958-9364    jamesd    jamesdpass   
4       emma.miller@x.dummyjson.com  +91 759-776-1614     emmaj     emmajpass   

   ...                                            address         macAddress  \
0  ...  {'address': '626 Main Street', 'city': 'Phoeni...  47:fa:41:18:ec:eb   
1  ...  {'address': '385 Fifth Str

## Transform API Structure

In [18]:
df_api = df_api.rename(columns={
    "id":"customer_id",
    "email":"email"
})



In [20]:
df_api["name"] = (
    df_api["firstName"].fillna("").astype(str)
    + ""
    +df_api["lastName"].fillna("").astype(str)
).str.strip()

In [22]:
df_api["city"] = df_api["address"].apply(
    lambda x:x.get("city") if isinstance(x,dict) else None
)

In [23]:
# This demo API does not provide updated_at.
df_api["updated_at"] = "2026-01-01 00:00:00"

In [25]:
df_api = df_api[
    ["customer_id", "name", "email", "city", "updated_at"]
]

In [26]:
print(df_api.head())

   customer_id             name                             email  \
0            1     EmilyJohnson     emily.johnson@x.dummyjson.com   
1            2  MichaelWilliams  michael.williams@x.dummyjson.com   
2            3      SophiaBrown      sophia.brown@x.dummyjson.com   
3            4       JamesDavis       james.davis@x.dummyjson.com   
4            5       EmmaMiller       emma.miller@x.dummyjson.com   

           city           updated_at  
0       Phoenix  2026-01-01 00:00:00  
1       Houston  2026-01-01 00:00:00  
2    Washington  2026-01-01 00:00:00  
3       Seattle  2026-01-01 00:00:00  
4  Jacksonville  2026-01-01 00:00:00  


### Combine data from multiple sources

Now we have:
- CSV data in df_csv.
- API data in df_api.

combine them into one DataFrame.

In [27]:
df = pd.concat([df_csv,df_api],ignore_index=True)

In [30]:
print(df.head(10))
print("Total Records",len(df))

   customer_id     name              email       city           updated_at
0          101    Rahul    rahul@gmail.com       Pune  2026-11-01 10:00:00
1          102  Atharva  atharva@gmail.com     Mumbai  2026-11-01 11:00:00
2          103    Mohit    mohit@gmail.com      Delhi  2026-11-02 12:00:00
3          104    Priya    priya@gmail.com  Bangalore  2026-11-02 13:00:00
4          105     Amit     amit@gmail.com  Hyderabad  2026-11-02 14:00:00
5          106    Sneha    sneha@gmail.com    Chennai  2026-11-02 15:00:00
6          107    Rohit    rohit@gmail.com    Kolkata  2026-11-03 09:00:00
7          108     Neha     neha@gmail.com     Jaipur  2026-11-03 10:00:00
8          109    Karan    karan@gmail.com  Ahmedabad  2026-11-03 11:00:00
9          110    Pooja    pooja@gmail.com      Surat  2026-11-03 12:00:00
Total Records 60


## Transform the data

In [33]:
df = df.dropna(subset=["customer_id"])

# Fill Missing Values

df["name"]=df["name"].fillna("Unknown")
df["email"]=df["email"].fillna("unknown@example.com")
df["city"]=df["city"].fillna("Unknown")

In [34]:
print(df.isnull().sum())

customer_id    0
name           0
email          0
city           0
updated_at     0
dtype: int64


# Change data types

In [36]:
df["customer_id"] = pd.to_numeric(
    df["customer_id"]
).astype("int64")

df["updated_at"]=pd.to_datetime(
    df["updated_at"]
) 


# Remove Records with invalid timestamps

In [39]:
# Remove records with invalid timestamps
df = df.dropna(subset=["updated_at"])

# Clean text fields
df["name"] = df["name"].astype(str).str.strip()

df["email"] = df["email"].astype(str).str.strip().str.lower()

df["city"] = df["city"].astype(str).str.strip()

print(df.dtypes)

customer_id             int64
name                      str
email                     str
city                      str
updated_at     datetime64[us]
dtype: object


# Apply business rules

In [42]:
# Remove the duplicate customer Id within the batch

df = df.sort_values("updated_at")

df = df.drop_duplicates(
    subset=["customer_id"],
    keep="last"
)

df = df[
    df["email"].str.contains("@",na=False)
]

df=df.reset_index(drop=True)

print("Cleaned Data: ")
print(df)


Cleaned Data: 
    customer_id              name                              email  \
0            10  IsabellaAnderson  isabella.anderson@x.dummyjson.com   
1             8         AvaTaylor         ava.taylor@x.dummyjson.com   
2             7    AlexanderJones    alexander.jones@x.dummyjson.com   
3             6      OliviaWilson      olivia.wilson@x.dummyjson.com   
4             5        EmmaMiller        emma.miller@x.dummyjson.com   
5             4        JamesDavis        james.davis@x.dummyjson.com   
6             3       SophiaBrown       sophia.brown@x.dummyjson.com   
7             2   MichaelWilliams   michael.williams@x.dummyjson.com   
8             1      EmilyJohnson      emily.johnson@x.dummyjson.com   
9             9     EthanMartinez     ethan.martinez@x.dummyjson.com   
10          101             Rahul                    rahul@gmail.com   
11          102           Atharva                  atharva@gmail.com   
12          103             Mohit                

In [43]:
# Created a Staging Table for it 

# CREATE TABLE IF NOT EXISTS staging_customers (

#     customer_id INT PRIMARY KEY,
#     name VARCHAR(100),
#     email VARCHAR(150),
#     city VARCHAR(100),
#     updated_at DATETIME
# )

In [ ]:
# INSERT INTO customers (
#     customer_id,
#     name,
#     email,
#     city,
#     updated_at
# )
# SELECT
#     s.customer_id,
#     s.name,
#     s.email,
#     s.city,
#     s.updated_at
# FROM staging_customers s
# WHERE NOT EXISTS (
#     SELECT 1
#     FROM customers c
#     WHERE c.customer_id = s.customer_id
# );


In [44]:
# Load cleaned data into staging table

In [55]:
df.to_sql(
    name="staging_customers",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Data loaded into staging table!")

Data loaded into staging table!


In [56]:
# Load data into the main table using UPSERT
# This is the most important code in our project.
# We will use MySQL's ON DUPLICATE KEY UPDATE.

In [60]:
upsert_query = """
INSERT INTO customers (
    customer_id,
    name,
    email,
    city,
    updated_at
)

SELECT
    s.customer_id,
    s.name,
    s.email,
    s.city,
    s.updated_at

FROM staging_customers AS s

ON DUPLICATE KEY UPDATE

    name = IF(
        VALUES(updated_at) >= customers.updated_at,
        VALUES(name),
        customers.name
    ),

    email = IF(
        VALUES(updated_at) >= customers.updated_at,
        VALUES(email),
        customers.email
    ),

    city = IF(
        VALUES(updated_at) >= customers.updated_at,
        VALUES(city),
        customers.city
    ),

    updated_at = GREATEST(
        customers.updated_at,
        VALUES(updated_at)
    )
"""

In [61]:
with engine.begin() as connection:

    connection.execute(text(upsert_query))

print("UPSERT completed successfully!")

UPSERT completed successfully!


In [62]:
df_result = pd.read_sql(
    "SELECT * FROM customers ORDER BY customer_id",
    engine
)

display(df_result)

,customer_id,name,email,city,updated_at,created_at
0,1,EmilyJohnson,emily.johnson@x.dummyjson.com,Phoenix,2026-01-01 00:00:00,2026-11-03 17:52:32
1,2,MichaelWilliams,michael.williams@x.dummyjson.com,Houston,2026-01-01 00:00:00,2026-11-03 17:52:32
2,3,SophiaBrown,sophia.brown@x.dummyjson.com,Washington,2026-01-01 00:00:00,2026-11-03 17:52:32
3,4,JamesDavis,james.davis@x.dummyjson.com,Seattle,2026-01-01 00:00:00,2026-11-03 17:52:32
4,5,EmmaMiller,emma.miller@x.dummyjson.com,Jacksonville,2026-01-01 00:00:00,2026-11-03 17:52:32
5,6,OliviaWilson,olivia.wilson@x.dummyjson.com,Fort Worth,2026-01-01 00:00:00,2026-11-03 17:52:32
6,7,AlexanderJones,alexander.jones@x.dummyjson.com,Indianapolis,2026-01-01 00:00:00,2026-11-03 17:52:32
7,8,AvaTaylor,ava.taylor@x.dummyjson.com,Fort Worth,2026-01-01 00:00:00,2026-11-03 17:52:32
8,9,EthanMartinez,ethan.martinez@x.dummyjson.com,San Antonio,2026-01-01 00:00:00,2026-11-03 17:52:32
9,10,IsabellaAnderson,isabella.anderson@x.dummyjson.com,New York,2026-01-01 00:00:00,2026-11-03 17:52:32


In [63]:
with engine.connect() as connection:

    count = connection.execute(
        text("SELECT COUNT(*) FROM customers")
    ).scalar()

print("Total Customers:", count)

Total Customers: 60


In [64]:
updated_data = {
    "customer_id": [102, 104],
    "name": ["Priya", "Sneha"],
    "email": ["priya@gmail.com", "sneha@gmail.com"],
    "city": ["Bangalore", "Pune"],
    "updated_at": [
        "2026-11-03 10:00:00",
        "2026-11-03 11:00:00"
    ]
}

df_updated = pd.DataFrame(updated_data)

df_updated["customer_id"] = df_updated["customer_id"].astype("int64")

df_updated["updated_at"] = pd.to_datetime(
    df_updated["updated_at"]
)

In [65]:
df_updated.to_sql(
    name="staging_customers",
    con=engine,
    if_exists="replace",
    index=False
)

print("Updated source data loaded!")

Updated source data loaded!


In [66]:
with engine.begin() as connection:

    connection.execute(text(upsert_query))

print("Updated records processed!")

Updated records processed!


In [67]:
df_result = pd.read_sql(
    "SELECT * FROM customers ORDER BY customer_id",
    engine
)

print(df_result)

    customer_id              name                              email  \
0             1      EmilyJohnson      emily.johnson@x.dummyjson.com   
1             2   MichaelWilliams   michael.williams@x.dummyjson.com   
2             3       SophiaBrown       sophia.brown@x.dummyjson.com   
3             4        JamesDavis        james.davis@x.dummyjson.com   
4             5        EmmaMiller        emma.miller@x.dummyjson.com   
5             6      OliviaWilson      olivia.wilson@x.dummyjson.com   
6             7    AlexanderJones    alexander.jones@x.dummyjson.com   
7             8         AvaTaylor         ava.taylor@x.dummyjson.com   
8             9     EthanMartinez     ethan.martinez@x.dummyjson.com   
9            10  IsabellaAnderson  isabella.anderson@x.dummyjson.com   
10          101             Rahul                    rahul@gmail.com   
11          102             Priya                    priya@gmail.com   
12          103             Mohit                    mohit@gmail